# 02 - Data Cleaning and Feature Preparation

This notebook documents the leakage decision for `Monthly_Salary`, builds the full and conservative feature sets, uses median/mode imputation in the preprocessing pipeline, and creates department profile tables for the counterfactual assignment step.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Markdown, display

current = Path.cwd().resolve()
candidate_roots = [current, current.parent, current.parent.parent]
ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'smartassign_pipeline.py').exists():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError('Could not locate the repository root.')

sys.path.insert(0, str(ROOT / 'src'))
from smartassign_pipeline import (
    PROCESSED_DATA_DIR,
    RESULTS_DIR,
    basic_data_overview,
    compute_department_profiles,
    ensure_output_directories,
    get_counterfactual_profile_columns,
    get_feature_sets,
    load_raw_data,
)

ensure_output_directories()
pd.set_option('display.max_columns', None)

df = load_raw_data()
overview = basic_data_overview(df)
feature_sets = get_feature_sets(df)

print(f'Shape: {overview["shape"]}')
print('Total missing values:', int(df.isna().sum().sum()))
print('Missing-value handling: median imputation for numeric columns and most-frequent imputation for categorical columns inside the model pipeline.')

cleaned_df = df.copy()
cleaned_df['Hire_Date'] = pd.to_datetime(cleaned_df['Hire_Date'], errors='coerce')
cleaned_df.to_csv(PROCESSED_DATA_DIR / 'cleaned_employee_data.csv', index=False)

print('Full feature set:')
print(feature_sets['full'])
print('\nConservative feature set:')
print(feature_sets['conservative'])

for feature_set_name, feature_columns in feature_sets.items():
    profile_columns = get_counterfactual_profile_columns(feature_set_name, df)
    department_profiles = compute_department_profiles(df, profile_columns)
    profile_path = PROCESSED_DATA_DIR / f'department_profiles_{feature_set_name}.csv'
    department_profiles.to_csv(profile_path, index=False)
    print(f'\nDepartment profiles for {feature_set_name}:')
    display(department_profiles)
    print(f'Counterfactual profile columns used for {feature_set_name}: {profile_columns}')

metadata = {
    'leakage_feature_removed': 'Monthly_Salary',
    'excluded_columns_from_modeling': ['Employee_ID', 'Monthly_Salary', 'Performance_Score', 'Resigned', 'Hire_Date'],
    'encoding_choice': 'One-hot encoding for Department, Job_Title, Gender, and Education_Level because all have low cardinality in this dataset.',
    'train_test_split': '80/20 split stratified by Department.',
}
(RESULTS_DIR / 'data_prep_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')

summary_lines = [
    'Missing values are handled in the preprocessing pipeline with median imputation for numeric columns and most-frequent imputation for categorical columns.',
    'Monthly_Salary is removed from all modeling features because it is a documented leakage variable correlated with Performance_Score.',
    'Two feature sets are created: a full set that keeps Projects_Handled, Promotions, and Sick_Days, and a conservative set that drops them.',
    'Department and Job_Title are encoded with one-hot encoding because their cardinality is low in this dataset.',
    'Department profile medians are saved for the counterfactual assignment step, and the selection-bias assumption is documented explicitly.',
]
display(Markdown('### Data Prep Summary\n' + '\n'.join(f'- {line}' for line in summary_lines)))

/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


Shape: (100000, 20)
Total missing values: 0
Missing-value handling: median imputation for numeric columns and most-frequent imputation for categorical columns inside the model pipeline.
Full feature set:
['Department', 'Job_Title', 'Gender', 'Education_Level', 'Age', 'Years_At_Company', 'Work_Hours_Per_Week', 'Overtime_Hours', 'Remote_Work_Frequency', 'Team_Size', 'Training_Hours', 'Employee_Satisfaction_Score', 'Projects_Handled', 'Promotions', 'Sick_Days']

Conservative feature set:
['Department', 'Job_Title', 'Gender', 'Education_Level', 'Age', 'Years_At_Company', 'Work_Hours_Per_Week', 'Overtime_Hours', 'Remote_Work_Frequency', 'Team_Size', 'Training_Hours', 'Employee_Satisfaction_Score']

Department profiles for full:


,Department,Years_At_Company,Work_Hours_Per_Week,Overtime_Hours,Remote_Work_Frequency,Team_Size,Training_Hours,Employee_Satisfaction_Score,Projects_Handled,Promotions,Sick_Days,department_size
0,Customer Support,4.0,45.0,14.0,50.0,10.0,49.0,2.96,24.0,1.0,7.0,11116
1,Engineering,4.0,45.0,15.0,50.0,10.0,50.0,2.99,24.0,1.0,7.0,10956
2,Finance,4.0,45.0,14.0,50.0,10.0,50.0,2.99,24.0,1.0,7.0,11200
3,HR,4.0,45.0,14.0,50.0,10.0,49.0,3.02,24.0,1.0,7.0,10960
4,IT,4.0,45.0,15.0,50.0,10.0,49.0,3.03,24.0,1.0,7.0,11131
5,Legal,4.0,45.0,15.0,50.0,10.0,50.0,2.98,24.0,1.0,7.0,11118
6,Marketing,5.0,45.0,14.0,50.0,10.0,49.0,2.98,25.0,1.0,7.0,11216
7,Operations,4.0,45.0,14.0,50.0,10.0,48.0,3.03,24.0,1.0,7.0,11181
8,Sales,4.0,45.0,15.0,50.0,10.0,49.0,2.99,25.0,1.0,7.0,11122


Counterfactual profile columns used for full: ['Years_At_Company', 'Work_Hours_Per_Week', 'Overtime_Hours', 'Remote_Work_Frequency', 'Team_Size', 'Training_Hours', 'Employee_Satisfaction_Score', 'Projects_Handled', 'Promotions', 'Sick_Days']

Department profiles for conservative:


,Department,Years_At_Company,Work_Hours_Per_Week,Overtime_Hours,Remote_Work_Frequency,Team_Size,Training_Hours,Employee_Satisfaction_Score,department_size
0,Customer Support,4.0,45.0,14.0,50.0,10.0,49.0,2.96,11116
1,Engineering,4.0,45.0,15.0,50.0,10.0,50.0,2.99,10956
2,Finance,4.0,45.0,14.0,50.0,10.0,50.0,2.99,11200
3,HR,4.0,45.0,14.0,50.0,10.0,49.0,3.02,10960
4,IT,4.0,45.0,15.0,50.0,10.0,49.0,3.03,11131
5,Legal,4.0,45.0,15.0,50.0,10.0,50.0,2.98,11118
6,Marketing,5.0,45.0,14.0,50.0,10.0,49.0,2.98,11216
7,Operations,4.0,45.0,14.0,50.0,10.0,48.0,3.03,11181
8,Sales,4.0,45.0,15.0,50.0,10.0,49.0,2.99,11122


Counterfactual profile columns used for conservative: ['Years_At_Company', 'Work_Hours_Per_Week', 'Overtime_Hours', 'Remote_Work_Frequency', 'Team_Size', 'Training_Hours', 'Employee_Satisfaction_Score']


### Data Prep Summary
- Missing values are handled in the preprocessing pipeline with median imputation for numeric columns and most-frequent imputation for categorical columns.
- Monthly_Salary is removed from all modeling features because it is a documented leakage variable correlated with Performance_Score.
- Two feature sets are created: a full set that keeps Projects_Handled, Promotions, and Sick_Days, and a conservative set that drops them.
- Department and Job_Title are encoded with one-hot encoding because their cardinality is low in this dataset.
- Department profile medians are saved for the counterfactual assignment step, and the selection-bias assumption is documented explicitly.